In [10]:
import tensorflow as tf
from src.dataset import build_yolo_dataset
import numpy as np
from src.constants import ANCHORS, GRID_SIZE, IMAGE_SIZE
from src.model import build_model

In [99]:
val_dataset = build_yolo_dataset(batch_size= 16, split= "train")

In [100]:
for batch in val_dataset.take(1):
    images = batch["image"]
    true_boxes = batch["boxes"]
    true_labels = batch["labels"]
    targets = batch["targets"]
print("Images shape:", images.shape)
print("Targets shape:", targets.shape)

Images shape: (16, 224, 224, 3)
Targets shape: (16, 7, 7, 2, 25)


In [61]:
model = build_model()
checkpoint = tf.train.Checkpoint(model=model)
checkpoint.restore(
    "checkpoints/best_model-35"
).expect_partial()
preds = model(images, training=False)
print("Predictions shape:", preds.shape)

Predictions shape: (16, 7, 7, 2, 25)


In [125]:
image = batch["image"][3]
pred = preds[3]

grid_h, grid_w = GRID_SIZE

detections = []

for cell_y in range(grid_h):
    for cell_x in range(grid_w):
        for anchor_idx in range(len(ANCHORS)):

            raw_objectness = pred[cell_y, cell_x, anchor_idx, 0]
            tx = pred[cell_y, cell_x, anchor_idx, 1]
            ty = pred[cell_y, cell_x, anchor_idx, 2]
            tw = pred[cell_y, cell_x, anchor_idx, 3]
            th = pred[cell_y, cell_x, anchor_idx, 4]
            raw_class_logits = pred[cell_y, cell_x, anchor_idx, 5:]

            # Decode center
            cx = (tx + cell_x) / grid_w
            cy = (ty + cell_y) / grid_h

            # Decode width/height
            anchor_w, anchor_h = ANCHORS[anchor_idx]

            bw = np.exp(tw) * anchor_w
            bh = np.exp(th) * anchor_h

            # Convert to corners
            xmin = cx - bw / 2
            ymin = cy - bh / 2
            xmax = cx + bw / 2
            ymax = cy + bh / 2

            # Convert normalized coordinates to pixels
            xmin *= IMAGE_SIZE[1]
            ymin *= IMAGE_SIZE[0]
            xmax *= IMAGE_SIZE[1]
            ymax *= IMAGE_SIZE[0]

            objectness = tf.sigmoid(raw_objectness)
            class_probs = tf.nn.softmax(raw_class_logits)

            class_id = np.argmax(class_probs)
            class_prob = class_probs[class_id]

            confidence = objectness * class_prob

            detections.append({
               "box": [
               float(xmin),
               float(ymin),
               float(xmax),
               float(ymax)
                 ],
              "objectness": float(objectness),
              "class_id": int(class_id),
              "class_prob": float(class_prob),
              "confidence": float(confidence),
             })

In [126]:
CONF_THRESHOLD = 0.05

filtered_detections = [
    d for d in detections
    if d["confidence"] >= CONF_THRESHOLD
]

print("Total predictions:", len(detections))
print("After confidence filtering:", len(filtered_detections))

for d in filtered_detections:
    d["box"] = [
        max(0, min(224, x))
        for x in d["box"]
    ]

Total predictions: 98
After confidence filtering: 1


In [127]:
for d in sorted(
    detections,
    key=lambda x: x["confidence"],
    reverse=True
):
    if d["confidence"] >= 0.05:
        print(
            f"class={d['class_id']}, "
            f"confidence={d['confidence']:.3f}, "
            f"objectness={d['objectness']:.3f}, "
            f"class_prob={d['class_prob']:.3f}, "
            f"box={d['box']}"
        )

class=14, confidence=0.998, objectness=0.999, class_prob=0.999, box=[14.502394676208496, 32.719749450683594, 146.85031127929688, 218.8570098876953]


In [128]:
print("Ground truth:")

for box, label in zip(
    true_boxes[3].numpy(),
    true_labels[3].numpy()
):
    print(
        f"class={int(label)}, "
        f"box={box}"
    )

Ground truth:
class=14, box=[ 14.933332  28.224    147.54132  224.      ]
class=0, box=[0. 0. 0. 0.]
class=0, box=[0. 0. 0. 0.]
class=0, box=[0. 0. 0. 0.]
class=0, box=[0. 0. 0. 0.]
class=0, box=[0. 0. 0. 0.]
class=0, box=[0. 0. 0. 0.]


In [129]:
target = targets[3]

positive_indices = tf.where(target[..., 0] == 1)

print("Positive locations:")
print(positive_indices.numpy())

Positive locations:
[[3 2 1]]


In [130]:
for cell_y, cell_x, anchor_idx in positive_indices.numpy():

    target_values = target[cell_y, cell_x, anchor_idx]

    true_class = int(tf.argmax(target_values[5:]))

    print(
        f"Cell=({cell_x}, {cell_y}), "
        f"Anchor={anchor_idx}"
    )

    print("  objectness:", float(target_values[0]))
    print("  tx:", float(target_values[1]))
    print("  ty:", float(target_values[2]))
    print("  tw:", float(target_values[3]))
    print("  th:", float(target_values[4]))
    print("  class:", true_class)

Cell=(2, 3), Anchor=1
  objectness: 1.0
  tx: 0.5386664867401123
  ty: 0.94100022315979
  tw: 1.0851892232894897
  th: 1.7624449729919434
  class: 14


In [131]:
for cell_y, cell_x, anchor_idx in positive_indices.numpy():

    target_values = target[cell_y, cell_x, anchor_idx]
    pred_values = pred[cell_y, cell_x, anchor_idx]

    pred_objectness = float(tf.sigmoid(pred_values[0]))

    pred_class_probs = tf.nn.softmax(pred_values[5:])
    pred_class = int(tf.argmax(pred_class_probs))
    pred_class_prob = float(pred_class_probs[pred_class])

    true_class = int(tf.argmax(target_values[5:]))

    print(f"\nCell=(x={cell_x}, y={cell_y}), Anchor={anchor_idx}")
    print(f"True class: {true_class}")
    print(f"Pred class: {pred_class}")
    print(f"Pred class probability: {pred_class_prob:.3f}")

    print(f"\nObjectness:")
    print(f"  target:     {float(target_values[0]):.3f}")
    print(f"  prediction: {pred_objectness:.3f}")

    print("\nBox:")
    print(f"  tx: target={float(target_values[1]):.3f}, "
          f"pred={float(pred_values[1]):.3f}")
    print(f"  ty: target={float(target_values[2]):.3f}, "
          f"pred={float(pred_values[2]):.3f}")
    print(f"  tw: target={float(target_values[3]):.3f}, "
          f"pred={float(pred_values[3]):.3f}")
    print(f"  th: target={float(target_values[4]):.3f}, "
          f"pred={float(pred_values[4]):.3f}")


Cell=(x=2, y=3), Anchor=1
True class: 14
Pred class: 14
Pred class probability: 0.999

Objectness:
  target:     1.000
  prediction: 0.999

Box:
  tx: target=0.539, pred=0.521
  ty: target=0.941, pred=0.931
  tw: target=1.085, pred=1.083
  th: target=1.762, pred=1.712


In [33]:
from src.loss import YOLOLoss
model_test = build_model()

loss_fn_test = YOLOLoss(
    lambda_box=5.0,
    lambda_obj=1.0,
    lambda_noobj=0.5,
    lambda_class=1.0
)

In [ ]:
images_test = images[:1]
targets_test = targets[:1]

with tf.GradientTape() as tape:
    predictions_test = model_test(images_test, training=True)

    loss, box_loss, obj_loss, no_obj_loss, class_loss = \
        loss_fn_test.compute_losses_components(
            targets_test,
            predictions_test
        )

gradients = tape.gradient(loss, model_test.trainable_variables)

print("Loss:", float(loss))
print("Box:", float(box_loss))
print("Obj:", float(obj_loss))
print("NoObj:", float(no_obj_loss))
print("Class:", float(class_loss))

print("\nGradient check:")
for variable, gradient in zip(model_test.trainable_variables, gradients):
    print(
        variable.name,
        "gradient is None:", gradient is None
    )

Loss: 116.04615783691406
Box: 11.16722297668457
Obj: 0.2837398052215576
NoObj: 112.41119384765625
Class: 3.7207069396972656

Gradient check:
kernel gradient is None: False
gamma gradient is None: False
beta gradient is None: False
kernel gradient is None: False
gamma gradient is None: False
beta gradient is None: False
kernel gradient is None: False
gamma gradient is None: False
beta gradient is None: False
kernel gradient is None: False
gamma gradient is None: False
beta gradient is None: False
kernel gradient is None: False
kernel gradient is None: False
gamma gradient is None: False
beta gradient is None: False
gamma gradient is None: False
beta gradient is None: False
kernel gradient is None: False
gamma gradient is None: False
beta gradient is None: False
kernel gradient is None: False
gamma gradient is None: False
beta gradient is None: False
kernel gradient is None: False
gamma gradient is None: False
beta gradient is None: False
kernel gradient is None: False
kernel gradient is

In [101]:
single_image = images[:8]
single_target = targets[:8]


optimizer_test = tf.keras.optimizers.Adam(learning_rate=1e-3)

for step in range(1000):

    with tf.GradientTape() as tape:
        prediction = model_test(single_image, training=True)

        total_loss, box_loss, obj_loss, no_obj_loss, class_loss = \
            loss_fn_test.compute_losses_components(
                single_target,
                prediction
            )

    gradients = tape.gradient(
        total_loss,
        model_test.trainable_variables
    )

    optimizer_test.apply_gradients(
        zip(gradients, model_test.trainable_variables)
    )

    if step % 100 == 0:
        print(
            step,
            float(total_loss),
            float(box_loss),
            float(obj_loss),
            float(no_obj_loss),
            float(class_loss)
        )

0 210.6611785888672 38.00579833984375 10.688793182373047 2.075448989868164 8.90566349029541
100 0.8076590895652771 0.052879698574543 0.23726961016654968 0.5337925553321838 0.03909476846456528
200 0.21190336346626282 0.02269606664776802 0.034000858664512634 0.0946338027715683 0.017105277627706528
300 0.11257230490446091 0.01596427522599697 0.012388980947434902 0.025258848443627357 0.007732517085969448
400 0.06709884107112885 0.008781610056757927 0.006739849224686623 0.025080841034650803 0.0039105224423110485
500 0.044607069343328476 0.006686125881969929 0.003362247720360756 0.010523710399866104 0.0025523416697978973
600 0.07773401588201523 0.013612071983516216 0.0038158586248755455 0.007651303894817829 0.002032143995165825
700 0.03129681199789047 0.004443192854523659 0.002859721891582012 0.007638562470674515 0.0024018443655222654
800 0.026287337765097618 0.0039792973548173904 0.0018161400221288204 0.005596713162958622 0.0017763536889106035
900 0.014699007384479046 0.001990857068449259 0

In [102]:
preds = model_test(single_image, training=False)